# Validación Cruzada Estratificada para KGE

Implementación de un esquema de validación cruzada k-fold estratificada sobre grafos de conocimiento (KGs).  
La estratificación se realiza por **tipo de relación**, garantizando que cada fold mantenga la distribución relacional del dataset original.  

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from tabulate import tabulate
from google.colab import drive, files

## 2. Carga del dataset

Se monta Google Drive y se carga el dataset de tripletas `(sujeto, relación, objeto)` en formato CSV separado por `;`.  
Se imprime una muestra, el número total de tripletas y los nombres de columnas para verificar la carga.

In [ ]:
drive.mount('/content/drive', force_remount=True)
main_dir = '/content/drive/MyDrive/Datos'

file_path = f"{main_dir}/GoT.csv"
df = pd.read_csv(file_path, sep=';')

print(df.head())
print(df.shape)
print(df.columns)

## 3. Deduplicación de tripletas

Se eliminan filas duplicadas considerando las tres columnas `(sujeto, relación, objeto)`.  
Tripletas repetidas sesgarían la distribución de los folds y contaminarían la evaluación.

In [ ]:
n_antes = len(df)

df = df.drop_duplicates(
    subset=[df.columns[0], df.columns[1], df.columns[2]]
).reset_index(drop=True)

n_despues = len(df)
print(f"Tripletas antes de deduplicar : {n_antes}")
print(f"Tripletas después de deduplicar: {n_despues}")
print(f"Duplicados eliminados          : {n_antes - n_despues}")

## 4. Estrategia de etiquetado por relación

Cada tripleta recibe una **etiqueta de estrato** igual al tipo de relación (`r`).  
Esto permite que el particionamiento Round Robin distribuya uniformemente cada relación entre los k folds.

La función `label_entities_by_relation` (conservada para referencia) clasificaba entidades por heurística;  
se reemplazó por `triple_label_for_stratification`, que usa directamente el campo de relación.

In [ ]:
def label_entities_by_relation(subject, relation, obj):
    """Clasifica (sujeto, objeto) según el tipo semántico de la relación. Conservada para referencia."""
    rel_map = {
        "ALLIED_WITH": ("Casa", "Casa"),
        "BRANCH_OF":   ("Casa", "Casa"),
        "FOUNDED_BY":  ("Casa", "Persona"),
        "HEIR_TO":     ("Persona", "Persona"),
        "IN_REGION":   ("Casa", "Región"),
        "LED_BY":      ("Casa", "Persona"),
        "PARENT_OF":   ("Persona", "Persona"),
        "SEAT_OF":     ("Seat", "Casa"),
        "SPOUSE":      ("Persona", "Persona"),
        "SWORN_TO":    ("Casa", "Casa")
    }
    if relation in rel_map:
        return rel_map[relation]

    def heuristic_label(entity):
        if not isinstance(entity, str):
            return "Unknown"
        e = entity.strip()
        if e.startswith("House"):
            return "Casa"
        parts = e.split()
        if len(parts) == 0:
            return "Unknown"
        if parts[0].lower() == "the":
            return "Región"
        if len(parts) <= 3:
            all_cap = all(p and p[0].isupper() for p in parts)
            if all_cap:
                return "Persona"
        return "Unknown"

    return heuristic_label(subject), heuristic_label(obj)


def triple_label_for_stratification(subject, relation, obj):
    """Etiqueta de estrato: tipo de relación de la tripleta."""
    return relation


print("Distribución de relaciones (dataset deduplicado):")
print(df.iloc[:, 1].value_counts().to_string())

## 5. Particionamiento estratificado k-fold (Round Robin)

Las tripletas se agrupan por relación y se distribuyen en `k` folds mediante **Round Robin** tras barajado aleatorio reproducible.  
Esto garantiza que cada fold reciba aproximadamente `1/k` de las tripletas de cada tipo de relación.

In [ ]:
def create_stratified_partitions_by_triple(triples_df, k_folds=5, random_seed=42):
    np.random.seed(random_seed)
    triples = list(triples_df.itertuples(index=False, name=None))

    grouped       = defaultdict(list)
    triple_labels = {}

    for s, r, o in triples:
        lbl = triple_label_for_stratification(s, r, o)
        grouped[lbl].append((s, r, o))
        triple_labels[(s, r, o)] = lbl

    folds = [[] for _ in range(k_folds)]
    for lbl, tlist in grouped.items():
        tcopy = tlist.copy()
        np.random.shuffle(tcopy)
        for idx, triple in enumerate(tcopy):
            folds[idx % k_folds].append(triple)

    return folds, triple_labels


k_folds = 5
folds, triple_labels = create_stratified_partitions_by_triple(
    df, k_folds=k_folds, random_seed=42
)

fold_sizes = [(i + 1, len(folds[i])) for i in range(k_folds)]
print("Folds creados (cada tripleta en exactamente un fold):")
print(tabulate(fold_sizes, headers=["Fold", "Tamaño"], tablefmt="github"))

## 6. Verificación de los folds

Se comprueba que:
- **Ninguna tripleta aparece en más de un fold** (sin overlap exacto).
- La distribución de relaciones por fold es proporcional al dataset completo.
- Se reporta el número de entidades compartidas entre pares de folds (overlap de entidades es esperado en KGs).

In [ ]:
def fold_summary_and_checks(folds, triple_labels):
    summaries          = []
    fold_entities_sets = []
    fold_objects_sets  = []

    for i, fold in enumerate(folds):
        lbl_counts    = Counter([triple_labels[t] for t in fold])
        objects       = [o for (_, _, o) in fold]
        obj_counter   = Counter(objects)
        repeated_objs = [obj for obj, cnt in obj_counter.items() if cnt > 1]
        relations     = [r for (_, r, _) in fold]
        rel_counter   = Counter(relations)
        subjects      = [s for (s, _, _) in fold]
        entities      = set(subjects) | set(objects)

        summaries.append({
            "fold":                   i + 1,
            "by_label":               dict(lbl_counts),
            "repeated_objects_count": len(repeated_objs),
            "some_repeated_objects":  repeated_objs[:8],
            "unique_relations":       len(rel_counter),
            "relation_freq_sample":   list(rel_counter.items())[:8]
        })
        fold_entities_sets.append(entities)
        fold_objects_sets.append(set(objects))

    common_entities = []
    for i in range(len(folds)):
        for j in range(i + 1, len(folds)):
            commons_ent = fold_entities_sets[i] & fold_entities_sets[j]
            common_entities.append({
                "pair":         f"{i+1}-{j+1}",
                "common_count": len(commons_ent),
                "sample":       list(commons_ent)[:5]
            })

    seen     = {}
    overlaps = []
    for i, fold in enumerate(folds):
        for t in fold:
            if t in seen:
                overlaps.append({"triple": t, "first_fold": seen[t] + 1, "second_fold": i + 1})
            else:
                seen[t] = i

    return summaries, common_entities, overlaps


summaries, common_entities, overlaps = fold_summary_and_checks(folds, triple_labels)

print(f"Overlap exacto de triples entre folds: {len(overlaps)}")
if overlaps:
    print("  Aún hay duplicados — revisar el CSV original")
else:
    print("  Sin overlap — cada tripleta está en exactamente un fold")

print()
for s in summaries:
    print(f"Fold {s['fold']}:")
    for rel, cnt in sorted(s["by_label"].items()):
        print(f"  {rel:<15}: {cnt}")
    print(f"  Relaciones únicas: {s['unique_relations']}")

print()
print("Entidades comunes entre pares de folds (muestra):")
for ce in common_entities[:5]:
    print(f"  Par {ce['pair']}: {ce['common_count']} entidades comunes")

## 7. Exportar particiones a Excel

Cada fold se guarda en una hoja separada del archivo `.xlsx`, con columnas `Subject`, `Relation`, `Object` y `label`.  
El archivo se descarga automáticamente desde Colab.

In [ ]:
output_name = "folds_partitions_corregido.xlsx"
with pd.ExcelWriter(output_name) as writer:
    for i, fold in enumerate(folds, start=1):
        df_fold = pd.DataFrame(fold, columns=["Subject", "Relation", "Object"])
        df_fold["label"] = df_fold.apply(
            lambda r: triple_labels[(r["Subject"], r["Relation"], r["Object"])],
            axis=1
        )
        df_fold.to_excel(writer, sheet_name=f"Fold_{i}", index=False)

print("Guardado en:", output_name)
files.download(output_name)

## 8. Funciones de evaluación con ranking filtrado

`compute_filtered_ranks` calcula el rango de cada tripleta de prueba corrupting sujeto u objeto con todas las entidades del grafo, excluyendo tripletas verdaderas conocidas (**filtered setting** estándar en KGE).  
`compute_metrics_from_ranks` agrega los rangos en MRR, Hits@1, Hits@3 y Hits@10.

In [ ]:
def compute_filtered_ranks(test_triples, entities_list, score_fn, corrupt_side, filter_triples):
    ranks = []
    for s, r, o in test_triples:
        true_score       = score_fn(s, r, o)
        candidate_scores = []

        for entity in entities_list:
            if corrupt_side == "subject":
                corrupted = (entity, r, o)
            elif corrupt_side == "object":
                corrupted = (s, r, entity)
            else:
                continue

            if corrupted not in filter_triples:
                candidate_scores.append(score_fn(*corrupted))

        all_scores = sorted(candidate_scores + [true_score], reverse=True)
        ranks.append(all_scores.index(true_score) + 1)

    return ranks


def compute_metrics_from_ranks(ranks):
    if not ranks:
        return {"MRR": 0.0, "Hits@1": 0.0, "Hits@3": 0.0, "Hits@10": 0.0}

    mrr = hits1 = hits3 = hits10 = 0
    for rank in ranks:
        mrr += 1.0 / rank
        if rank <= 1:  hits1  += 1
        if rank <= 3:  hits3  += 1
        if rank <= 10: hits10 += 1

    n = len(ranks)
    return {
        "MRR":     mrr   / n,
        "Hits@1":  hits1 / n,
        "Hits@3":  hits3 / n,
        "Hits@10": hits10 / n
    }

## 9. Evaluación k-fold con modelo KGE

En cada iteración, los `k-1` folds restantes forman el conjunto de entrenamiento y el fold `i` es el conjunto de prueba.  
Al final se promedian las métricas de todos los folds.

> **Para conectar un modelo real:** reemplaza el `score_fn` placeholder por la función de puntuación del modelo entrenado.  
> Ejemplo con AmpliGraph (`TransE`):
> ```python
> from ampligraph.latent_features import TransE
> model = TransE(batches_count=100, epochs=200, k=100, eta=5,
>                optimizer='adam', optimizer_params={'lr': 1e-3},
>                loss='pairwise', regularizer='LP',
>                regularizer_params={'p': 3, 'lambda': 1e-5}, verbose=True)
> model.fit(np.array(train_triples))
> score_fn = lambda s, r, o: float(model.predict(np.array([[s, r, o]]))[0])
> ```

In [ ]:
entities_list   = list({e for fold in folds for t in fold for e in (t[0], t[2])})
all_triples_set = set(t for fold in folds for t in fold)

# Placeholder — reemplazar por model.predict() del modelo entrenado
score_fn = lambda s, r, o: float(np.random.rand())

results_by_fold = defaultdict(dict)

for i, test_fold in enumerate(folds, start=1):
    train_triples  = [t for j, f in enumerate(folds, start=1) if j != i for t in f]
    filter_triples = set(train_triples) | set(test_fold) | all_triples_set

    # model.fit(np.array(train_triples))  ← descomentar al usar modelo real

    ranks_head = compute_filtered_ranks(
        test_fold, entities_list, score_fn, corrupt_side="subject", filter_triples=filter_triples
    )
    ranks_tail = compute_filtered_ranks(
        test_fold, entities_list, score_fn, corrupt_side="object", filter_triples=filter_triples
    )

    metrics = compute_metrics_from_ranks(ranks_head + ranks_tail)
    results_by_fold[f"fold_{i}"] = metrics
    print(f"Fold {i} metrics:", metrics)

keys = list(next(iter(results_by_fold.values())).keys())
avg  = {k: float(np.mean([results_by_fold[f][k] for f in results_by_fold])) for k in keys}
print("\nPromedio entre folds:", avg)